# Collections Data Analyst Assignment

This notebook rebuilds the main business claim from the supplied raw data. The primary grain is one first targeting event per account-month. Successful payments are deduplicated by payment_id. August 2026 is incomplete and excluded from full-month trend conclusions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA=Path('../data')
REPORTS=Path('../reports')
payments=pd.read_csv(DATA/'payments.csv',parse_dates=['event_at'])
target=pd.read_csv(DATA/'daily_targeting.csv',parse_dates=['target_date'])
accounts=pd.read_csv(DATA/'accounts.csv')
borrowers=pd.read_csv(DATA/'borrowers.csv',parse_dates=['created_at','updated_at'])
calls=pd.read_csv(DATA/'calls.csv',parse_dates=['event_at'])
call_dispositions=pd.read_csv(DATA/'call_dispositions.csv',parse_dates=['event_at'])
promises=pd.read_csv(DATA/'promises_to_pay.csv',parse_dates=['event_at','promised_date'])
campaigns=pd.read_csv(DATA/'campaigns.csv',parse_dates=['start_at','end_at'])
agents=pd.read_csv(DATA/'agents.csv',parse_dates=['joined_at','updated_at'])

## 1. Payment integrity

Definition: a successful payment is one payment_status=SUCCESS row after deduplication by payment_id, keeping the latest event_at. payment_reference is not used as the transaction key because it is reused across accounts.

In [ ]:
success_raw=payments[payments.payment_status.eq('SUCCESS')].copy()
success=success_raw.sort_values(['payment_id','event_at']).drop_duplicates('payment_id',keep='last').copy()
summary=pd.DataFrame([{'raw_success_rows':len(success_raw),'unique_payment_ids':success.payment_id.nunique(),'raw_recovery_inr':success_raw.amount.sum(),'golden_recovery_inr':success.amount.sum(),'duplicate_amount_inr':success_raw.amount.sum()-success.amount.sum()}])
summary

## 2. Golden episode and one-payment-one-target attribution

The episode grain is one first target per account-month. The attribution table uses the same first-target population, so later targets cannot receive recovery that is absent from the golden episode table. Each payment is assigned to the latest eligible first target within the chosen window, so a payment is not counted twice.

In [ ]:
target['month']=target.target_date.dt.to_period('M').astype(str)
first_target=target.sort_values(['account_id','month','target_date','target_id']).drop_duplicates(['account_id','month'],keep='first')
P=success[['payment_id','account_id','event_at','amount']].sort_values(['event_at','account_id','payment_id'])
T=first_target[['target_id','account_id','target_date']].sort_values(['target_date','account_id','target_id'])
rows=[]
for days in [3,7,14,30]:
    matched=pd.merge_asof(P,T,left_on='event_at',right_on='target_date',by='account_id',direction='backward',tolerance=pd.Timedelta(days=days),allow_exact_matches=True).dropna(subset=['target_id'])
    rows.append({'window_days':days,'attributed_recovery_inr':matched.amount.sum(),'matched_payments':matched.payment_id.nunique(),'share_of_golden_recovery_pct':matched.amount.sum()/success.amount.sum()*100})
p.DataFrame(rows)

## 3. Monthly recovery and the 11% claim

In [ ]:
success['month']=success.event_at.dt.to_period('M').astype(str)
monthly=success.groupby('month').agg(recovery=('amount','sum'),successful_payments=('payment_id','nunique'),paid_accounts=('account_id','nunique')).reset_index()
monthly['mom_pct']=monthly.recovery.pct_change()*100
monthly['recovery_cr']=monthly.recovery/1e7
monthly

March is +11.03% versus February. The later full months do not sustain an 11% improvement. August is incomplete, so it is not used as a full-month comparator.

## 4. DPD mix standardization

In [ ]:
episodes=pd.read_csv(REPORTS/'golden_collection_episode.csv')
episodes['dpd_band']=pd.cut(episodes.dpd,[-1,7,30,60,90,10**9],labels=['0-7','8-30','31-60','61-90','90+'])
mix=episodes.groupby(['month','dpd_band'],observed=False).agg(targets=('account_id','nunique'),recovery=('recovery_7d','sum')).reset_index()
base=mix[mix.month.eq('2026-01')].set_index('dpd_band')
weights=base.targets/base.targets.sum()
adj=[]
for month,g in mix.groupby('month'):
    rates=g.set_index('dpd_band').recovery.div(g.set_index('dpd_band').targets.replace(0,np.nan))
    adj.append({'month':month,'raw':g.recovery.sum()/g.targets.sum(),'dpd_mix_adjusted':sum(rates.get(b,0)*weights.get(b,0) for b in weights.index)})
pd.DataFrame(adj)

If the mix-adjusted series moves similarly to the raw series, DPD composition alone is not a sufficient explanation. This is a standardization result, not a causal estimate.

## 5. Driver analysis

In [ ]:
channel=pd.read_csv(REPORTS/'channel_conversion_7d.csv')
dpd=pd.read_csv(REPORTS/'dpd_band_analysis.csv')
risk=pd.read_csv(REPORTS/'risk_segment_analysis.csv')
loan=pd.read_csv(REPORTS/'loan_type_analysis.csv')
geo=pd.read_csv(REPORTS/'geography_state_analysis.csv')
channel,dpd,risk,loan,geo

Channel conversion, geography, DPD, risk and loan type are descriptive. Channel differences are not causal because assignment is not randomized. Client and language are not present in the supplied schema, so they cannot be analysed. Agent tenure is not used for performance claims because all observed agent IDs have multiple joined dates and identity values.

## 6. Denominator and identity audit

In [ ]:
pd.read_csv(REPORTS/'denominator_audit.csv'),pd.read_csv(REPORTS/'identity_integrity.csv')

The primary denominator is unique account-months represented by first targets. Event borrower_id mismatches are retained as data-quality findings. Account-level metrics use account_id rather than blindly trusting event borrower_id.

## 7. Metric definitions and investment hurdle

In [ ]:
definitions=pd.read_csv(REPORTS/'metric_definitions.csv')
scenarios=pd.read_csv(REPORTS/'investment_scenarios.csv')
hurdle=pd.read_csv(REPORTS/'investment_hurdle.csv')
definitions,scenarios,hurdle

The ₹10 Cr decision cannot be supported by a causal ROI estimate from this observational dataset. The financial table is therefore a hurdle analysis. The break-even uplift is the minimum incremental annual recovery needed to recover ₹10 Cr. The recommended next step is a stratified randomized holdout.

## 8. Counterfactual design

Treatment: eligible accounts assigned to the new targeting strategy. Control: comparable eligible accounts retained on the previous strategy. Stratify by DPD, risk, loan type, prior recovery and geography. Primary outcome: 30-day golden recovery per eligible account. Scale only if the confidence interval for incremental recovery clears the break-even hurdle. This is an experiment design, not a retrospective causal claim.